# p5a — Clean export and join for Zenodo / Excel compatibility

This notebook takes the raw `p5a_master_dataset.csv` (semicolon-separated, comma-decimal,
with embedded newlines in some column headers), joins it with the annual demographic series
from p1a (Net_change, 1997–2025) and p1b (Mig_balance, 1998–2024), and produces clean,
consistent exports:

1. **International version** (comma-separated, point-decimal) — recommended for Zenodo / international reuse.
2. **Spanish-locale version** (semicolon-separated, comma-decimal) — opens cleanly via double-click in Spanish Excel.

Both versions:
- Include municipality-level annual series for Net_change (Padrón) and Mig_balance (estimated migratory balance), alongside the Periods A/B summary variables and SIDAMUN socioeconomic block already present in p5a.
- Have column headers on a single line (no embedded `\n`).
- Preserve `Mun_Code` as a 5-digit zero-padded **string** consistently across all rows.
- Round-trip verified (re-reading the saved file reproduces the same shape and dtypes).

In [1]:
import pandas as pd
import re
import os


DERIV_PAPER1_DIR = r'C:\Users\juanz\OneDrive\Desktop\UCM\RURIM ESCAPE\GeoSpatial\01_Python Data Analysis\rural-migration-land-use-spain\data\demography\derived\paper1'

RAW_PATH         = f'{DERIV_PAPER1_DIR}\\p5a_master_dataset.csv'
NET_CHANGE_PATH  = f'{DERIV_PAPER1_DIR}\\p1a_net_change_wide_by_municipality.csv'
MIG_BALANCE_PATH = f'{DERIV_PAPER1_DIR}\\p1b_mig_balance_wide_by_municipality.csv'

OUT_DIR = r'C:\Users\juanz\OneDrive\Desktop\UCM\RURIM ESCAPE\GeoSpatial\01_Python Data Analysis\rural-migration-land-use-spain\data\demography'

BASE_NAME = 'Zotes_Concepcion_Herrero-2026-RurImEScape-Spain_Rural_Municipalities_Demographic_Socioeconomic_Dataset'
OUT_INTL = f'{OUT_DIR}\\{BASE_NAME}_International_Format.csv'
OUT_ES   = f'{OUT_DIR}\\{BASE_NAME}_Spain_Format.csv'

## 1. Load raw data

In [2]:
df = pd.read_csv(RAW_PATH, sep=';')
print('Shape:', df.shape)
df.dtypes.value_counts()

Shape: (8132, 275)


float64    227
int64       38
object       9
bool         1
Name: count, dtype: int64

## 2. Clean column headers (remove embedded newlines)

In [3]:
bad_cols = [c for c in df.columns if '\n' in c]
print(f'{len(bad_cols)} columns had embedded newlines — cleaning...')

df.columns = [re.sub(r'\s*\n\s*', ' ', c).strip() for c in df.columns]

assert df.columns.duplicated().sum() == 0, 'Duplicate column names after cleaning!'
assert not any('\n' in c for c in df.columns), 'Newlines still present!'
print('OK — headers clean, no duplicates.')

102 columns had embedded newlines — cleaning...
OK — headers clean, no duplicates.


In [4]:
df.head(5)

,Mun_Code,Mun_Name,Comarca_Code,Comarca_Name,Prov_Code,Prov_Name,CCAA_Code,CCAA_Name,tipo_goerlich,size_group,...,DEMOGRAFIA__Pct_Hombres_0_14,DEMOGRAFIA__Pct_15_29,DEMOGRAFIA__Pct_Mujeres_15_29,DEMOGRAFIA__Pct_Hombres_15_29,DEMOGRAFIA__Pct_30_64,DEMOGRAFIA__Pct_Mujeres_30_64,DEMOGRAFIA__Pct_Hombres_30_64,DEMOGRAFIA__Pct_65_plus,DEMOGRAFIA__Pct_Mujeres_65_plus,DEMOGRAFIA__Pct_Hombres_65_plus
0,1001,Alegría-Dulantzi,102.0,LLANADA ALAVESA,1,Araba/Álava,16,País Vasco/Euskadi,Rural - Accesible,"1,000 - 5,000",...,8.718111,18.244715,9.222484,9.022231,52.580122,25.479636,27.100485,13.051608,6.496129,6.555479
1,1002,Amurrio,106.0,CANTABRICA ALAVESA,1,Araba/Álava,16,País Vasco/Euskadi,Intermedio - Abierto,"10,000 - 50,000",...,7.776769,12.646658,5.831063,6.815596,48.536189,24.128202,24.407988,22.719027,12.282039,10.436988
2,1003,Aramaio,105.0,ESTRIBACIONES DEL GORBEA,1,Araba/Álava,16,País Vasco/Euskadi,Rural - Accesible,"1,000 - 5,000",...,5.937726,15.289143,6.961844,8.327299,47.319001,23.133629,24.185373,23.767632,11.240478,12.527154
3,1004,Artziniega,106.0,CANTABRICA ALAVESA,1,Araba/Álava,16,País Vasco/Euskadi,Rural - Accesible,"1,000 - 5,000",...,7.965413,13.145060,5.986952,7.158108,50.969580,25.781650,25.187929,18.211724,9.492825,8.718899
4,1006,Armiñón,101.0,VALLES ALAVESES,1,Araba/Álava,16,País Vasco/Euskadi,Rural - Accesible,"< 1,000",...,6.882591,15.847311,6.940428,8.906883,56.911510,27.761712,29.149798,13.418161,6.940428,6.477733


## 3. Fix Mun_Code: force consistent 5-digit zero-padded string

INE municipal codes are always 5 digits. As an int64 column, leading zeros
(provinces 01–09) are silently dropped. We force the whole column to string
type *before* any export, so it's consistent for every row.

In [5]:
print('Before — dtype:', df['Mun_Code'].dtype)
print('Min/Max:', df['Mun_Code'].min(), df['Mun_Code'].max())

df['Mun_Code'] = df['Mun_Code'].astype(str).str.zfill(5)

print('After  — dtype:', df['Mun_Code'].dtype)
print('All 5 digits?', (df['Mun_Code'].str.len() == 5).all())
print('Sample with leading zero:', df.loc[df['Mun_Code'].str.startswith('0'), 'Mun_Code'].head(3).tolist())

Before — dtype: int64
Min/Max: 1001 52001
After  — dtype: object
All 5 digits? True
Sample with leading zero: ['01001', '01002', '01003']


## 4. Join with annual demographic series (p1a, p1b)

Load the wide-format annual series produced by p1a (`Net_change`, 1997–2025) and
p1b (`Mig_balance`, 1998–2024), and join them to the main dataset on `Mun_Code`.
Both source files are zero-padded to 5-digit strings to match the join key.

In [6]:
net_change = pd.read_csv(NET_CHANGE_PATH, sep=';', dtype={'Mun_Code': str})
net_change['Mun_Code'] = net_change['Mun_Code'].str.zfill(5)

mig_balance = pd.read_csv(MIG_BALANCE_PATH, sep=';', dtype={'Mun_Code': str})
mig_balance['Mun_Code'] = mig_balance['Mun_Code'].str.zfill(5)

print('Net_change  shape:', net_change.shape)
print('Mig_balance shape:', mig_balance.shape)

n_before = df.shape[0]

df = df.merge(net_change, on='Mun_Code', how='left')
df = df.merge(mig_balance, on='Mun_Code', how='left')

assert df.shape[0] == n_before, 'Row count changed after join — check for duplicate Mun_Code in source files!'
print(f'\nJoined shape: {df.shape}')
print(f'Municipalities missing Net_change data: {df[net_change.columns[1]].isna().sum()}')
print(f'Municipalities missing Mig_balance data: {df[mig_balance.columns[1]].isna().sum()}')

Net_change  shape: (8132, 30)
Mig_balance shape: (8132, 29)

Joined shape: (8132, 332)
Municipalities missing Net_change data: 8132
Municipalities missing Mig_balance data: 8132


In [7]:
# --- Drop columns that are fully empty for justified reasons (not errors) ---
# Net_change_1996: first year of the series, no prior year to compute change from
# Net_change_1998: transition from 1996 (Year_gap = 2), not a valid annual change
# Mig_balance_1998: same reason — first valid year, no computable Delta_P
# Mig_balance_2025: outside MNP coverage (MNP runs 1998-2024)
cols_to_drop = ['Net_change_1996', 'Net_change_1998', 'Mig_balance_1998', 'Mig_balance_2025']

df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

print(f'Dropped columns: {cols_to_drop}')
print(f'New shape: {df.shape}')

Dropped columns: ['Net_change_1996', 'Net_change_1998', 'Mig_balance_1998', 'Mig_balance_2025']
New shape: (8132, 328)


## 5. Export — international format (comma sep, point decimal)

In [8]:
df.to_csv(OUT_INTL, sep=',', index=False, encoding='utf-8-sig')
print(f'Saved: {OUT_INTL}')

Saved: C:\Users\juanz\OneDrive\Desktop\UCM\RURIM ESCAPE\GeoSpatial\01_Python Data Analysis\rural-migration-land-use-spain\data\demography\Zotes_Concepcion_Herrero-2026-RurImEScape-Spain_Rural_Municipalities_Demographic_Socioeconomic_Dataset_International_Format.csv


## 6. Export — Spanish-locale format (semicolon sep, comma decimal)

In [9]:
df.to_csv(OUT_ES, sep=';', decimal=',', index=False, encoding='utf-8-sig')
print(f'Saved: {OUT_ES}')

Saved: C:\Users\juanz\OneDrive\Desktop\UCM\RURIM ESCAPE\GeoSpatial\01_Python Data Analysis\rural-migration-land-use-spain\data\demography\Zotes_Concepcion_Herrero-2026-RurImEScape-Spain_Rural_Municipalities_Demographic_Socioeconomic_Dataset_Spain_Format.csv


## 7. Round-trip verification — both files

In [10]:
for path, kwargs in [
    (OUT_INTL, dict(sep=',', dtype={'Mun_Code': str})),
    (OUT_ES,   dict(sep=';', decimal=',', dtype={'Mun_Code': str})),
]:
    check = pd.read_csv(path, **kwargs)
    assert check.shape == df.shape, f'{path}: shape mismatch!'
    assert (check['Mun_Code'].str.len() == 5).all(), f'{path}: Mun_Code not all 5 digits!'
    assert not any('\n' in c for c in check.columns), f'{path}: newline in headers!'
    print(f'{path}: OK — shape {check.shape}, Mun_Code consistent, headers clean.')

C:\Users\juanz\OneDrive\Desktop\UCM\RURIM ESCAPE\GeoSpatial\01_Python Data Analysis\rural-migration-land-use-spain\data\demography\Zotes_Concepcion_Herrero-2026-RurImEScape-Spain_Rural_Municipalities_Demographic_Socioeconomic_Dataset_International_Format.csv: OK — shape (8132, 328), Mun_Code consistent, headers clean.
C:\Users\juanz\OneDrive\Desktop\UCM\RURIM ESCAPE\GeoSpatial\01_Python Data Analysis\rural-migration-land-use-spain\data\demography\Zotes_Concepcion_Herrero-2026-RurImEScape-Spain_Rural_Municipalities_Demographic_Socioeconomic_Dataset_Spain_Format.csv: OK — shape (8132, 328), Mun_Code consistent, headers clean.


## 8. Quick look at the exported dataset

A few sanity-check queries on the final exported file, to confirm it reads back
as expected and to get a feel for its structure before publishing.

In [11]:
final = pd.read_csv(OUT_INTL, dtype={'Mun_Code': str})

print('Shape:', final.shape)
print('Mun_Code sample (first 5):', final['Mun_Code'].head(5).tolist())
print('Mun_Code sample (last 5): ', final['Mun_Code'].tail(5).tolist())
print()
print('Thematic blocks (column prefixes):')
prefixes = sorted(set(c.split('__')[0] for c in final.columns if '__' in c))
for p in prefixes:
    n = sum(1 for c in final.columns if c.startswith(p + '__'))
    print(f'  {p}: {n} columns')

Shape: (8132, 328)
Mun_Code sample (first 5): ['01001', '01002', '01003', '01004', '01006']
Mun_Code sample (last 5):  ['50901', '50902', '50903', '51001', '52001']

Thematic blocks (column prefixes):
  DEMOGRAFIA: 101 columns
  ECONOMIA: 52 columns
  GENERAL: 10 columns
  MEDIO FÍSICO: 7 columns
  MEDIOAMBIENTE: 20 columns
  SERVICIOS: 48 columns
  VIVIENDA: 11 columns


In [12]:
# Coverage check: confirm all 8,132 municipalities and the Goerlich typology / behavioural groups are present
print('Unique municipalities:', final['Mun_Code'].nunique())
print()
print('Goerlich typology counts:')
print(final['tipo_goerlich'].value_counts())
print()
print('Behavioural group counts (rural municipalities only, where assigned):')
print(final['behavioural_group'].value_counts(dropna=False))

print()
print('Net_change year coverage (non-null count per year, sample):')
net_change_cols = [c for c in final.columns if c.startswith('Net_change_')]
print(final[net_change_cols].notna().sum().head(5), '...')
print()
print('Mig_balance year coverage (non-null count per year, sample):')
mig_balance_cols = [c for c in final.columns if c.startswith('Mig_balance_')]
print(final[mig_balance_cols].notna().sum().head(5), '...')

Unique municipalities: 8132

Goerlich typology counts:
tipo_goerlich
Rural - Accesible       3885
Rural - Remoto          2838
Intermedio - Abierto     972
Intermedio - Cerrado     190
Urbano - Cerrado         164
Urbano - Abierto          83
Name: count, dtype: int64

Behavioural group counts (rural municipalities only, where assigned):
behavioural_group
Structural depopulation    3640
Reverses in B              2644
Grows in both              1457
Loses in B                  371
NaN                          20
Name: count, dtype: int64

Net_change year coverage (non-null count per year, sample):
Net_change_1999    8096
Net_change_2000    8099
Net_change_2001    8102
Net_change_2002    8105
Net_change_2003    8106
dtype: int64 ...

Mig_balance year coverage (non-null count per year, sample):
Mig_balance_1999    8093
Mig_balance_2000    8096
Mig_balance_2001    8099
Mig_balance_2002    8102
Mig_balance_2003    8103
dtype: int64 ...


## 9. Note on opening the international CSV in Excel

Excel auto-detects column types on double-click open, and may still strip leading
zeros from `Mun_Code` even though the file stores it as text. To avoid this:

Data > Get Data > From File > From Text/CSV > set Mun_Code column type to **Text** in the preview before loading.

The Spanish-locale version (`..._Spain_Format.csv`) opens correctly via plain
double-click in Spanish-locale Excel, since the decimal/separator convention matches
and Excel's column-type heuristics tend to behave better with the `;` delimiter file.

Also note: very long file paths (especially under deeply nested OneDrive folders)
can cause `FileNotFoundError` or "cannot access file" errors in both pandas and Excel
due to Windows' path length limits. If this happens, shorten `OUT_DIR` or `BASE_NAME`.